In [10]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Set project paths
DATA_DIR = Path("../data")
MODEL_DIR = Path("../model")
MODEL_DIR.mkdir(exist_ok=True)

print("Paths ready.")


Paths ready.


In [11]:
fake = pd.read_csv(DATA_DIR / "Fake.csv")
true = pd.read_csv(DATA_DIR / "True.csv")

fake["label"] = 1   # Fake
true["label"] = 0   # Real

df_us = pd.concat([fake, true], ignore_index=True)
df_us = df_us.sample(frac=1, random_state=42).reset_index(drop=True)

print("US Dataset:", df_us.shape)
df_us.head()


US Dataset: (44898, 5)


,title,text,subject,date,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",1
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",0
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",0
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",1
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",0


In [12]:
df_ind = pd.read_csv(DATA_DIR / "mostly_fake.csv")

df_ind["label"] = df_ind["label"].map({"REAL": 0, "FAKE": 1})
df_ind = df_ind[["text", "label"]]

print("Indian Fake/Real:", df_ind.shape)
df_ind.head()


Indian Fake/Real: (3729, 2)


,text,label
0,Payal has accused filmmaker Anurag Kashyap of ...,0
1,A four-minute-long video of a woman criticisin...,1
2,"Republic Poll, a fake Twitter account imitatin...",1
3,"Delhi teen finds place on UN green list, turns...",0
4,Delhi: A high-level meeting underway at reside...,0


In [13]:
df_india_real = pd.read_csv(
    DATA_DIR / "india-news-headlines.csv",
    usecols=["headline_text"],
    nrows=5000
)

df_india_real = df_india_real.rename(columns={"headline_text": "text"})
df_india_real["label"] = 0

print("Indian Real News:", df_india_real.shape)
df_india_real.head()


Indian Real News: (5000, 2)


,text,label
0,Status quo will not be disturbed at Ayodhya; s...,0
1,Fissures in Hurriyat over Pak visit,0
2,America's unwanted heading for India?,0
3,For bigwigs; it is destination Goa,0
4,Extra buses to clear tourist traffic,0


In [14]:
df_all = pd.concat([df_us, df_ind, df_india_real], ignore_index=True)
df_all = df_all.sample(frac=1, random_state=42).reset_index(drop=True)

print("Final merged dataset:", df_all.shape)
df_all.head()


Final merged dataset: (53627, 5)


,title,text,subject,date,label
0,Hillary Clinton also wants Britain to stay in ...,LONDON/WASHINGTON (Reuters) - U.S. Democratic ...,politicsNews,"April 23, 2016",0
1,Over Half Of The 23 NFL Players Still Kneeling...,Are you one bit surprised by the fact that the...,left-news,"Dec 18, 2017",1
2,Exclusive: Trump selects Washington lawyer Joe...,WASHINGTON (Reuters) - President Donald Trump ...,politicsNews,"October 18, 2017",0
3,THE STATE OF OUR NATION Is Perfectly Illustrat...,The 12 funniest halloween memes in no particul...,politics,"Oct 31, 2015",1
4,Arrest warrant for ex-Catalan leader 'normal' ...,MADRID (Reuters) - If the ousted Catalan leade...,worldnews,"November 2, 2017",0


In [15]:
def clean_text(text):
    if isinstance(text, str):
        text = text.lower()
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    return ""

df_all["clean_text"] = df_all["text"].apply(clean_text)
df_all.head()


,title,text,subject,date,label,clean_text
0,Hillary Clinton also wants Britain to stay in ...,LONDON/WASHINGTON (Reuters) - U.S. Democratic ...,politicsNews,"April 23, 2016",0,london washington reuters u s democratic presi...
1,Over Half Of The 23 NFL Players Still Kneeling...,Are you one bit surprised by the fact that the...,left-news,"Dec 18, 2017",1,are you one bit surprised by the fact that the...
2,Exclusive: Trump selects Washington lawyer Joe...,WASHINGTON (Reuters) - President Donald Trump ...,politicsNews,"October 18, 2017",0,washington reuters president donald trump has ...
3,THE STATE OF OUR NATION Is Perfectly Illustrat...,The 12 funniest halloween memes in no particul...,politics,"Oct 31, 2015",1,the funniest halloween memes in no particular ...
4,Arrest warrant for ex-Catalan leader 'normal' ...,MADRID (Reuters) - If the ousted Catalan leade...,worldnews,"November 2, 2017",0,madrid reuters if the ousted catalan leader ca...


In [16]:
from sklearn.model_selection import train_test_split

X = df_all["clean_text"]
y = df_all["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)


Train: (42901,) Test: (10726,)


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=7000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

X_train_tfidf.shape, X_test_tfidf.shape


((42901, 7000), (10726, 7000))

In [18]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=3000)
model.fit(X_train_tfidf, y_train)

print("Model trained successfully.")


Model trained successfully.


In [19]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.965970538877494

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.98      0.97      5654
           1       0.98      0.95      0.96      5072

    accuracy                           0.97     10726
   macro avg       0.97      0.97      0.97     10726
weighted avg       0.97      0.97      0.97     10726


Confusion Matrix:
 [[5541  113]
 [ 252 4820]]


In [20]:
def predict_news(text):
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    return "FAKE NEWS ❌" if pred == 1 else "REAL NEWS ✔️"

print(predict_news("Government of India releases new cybersecurity guidelines."))
print(predict_news("NASA confirms Earth will be dark for 15 days straight!"))


REAL NEWS ✔️
REAL NEWS ✔️


In [21]:
import joblib

joblib.dump(model, "../model/fake_news_model.pkl")
joblib.dump(vectorizer, "../model/tfidf_vectorizer.pkl")

print("Model & vectorizer saved successfully!")


Model & vectorizer saved successfully!
